#### 常數

In [1]:
POSE_TYPE = "twist"
POSE_TAG = "close_30"

JSON_FILE = f"./json/{POSE_TYPE}/{POSE_TAG}.json"
SAVE_FOLDER = f"./output/analyze/{POSE_TYPE}/"
SAVE_ANGLE_PLOT_NAME = f"{POSE_TAG}_angle_plot.png"
SAVE_POSES_PLOT_NAME = f"{POSE_TAG}_poses.png"

#### 套件

In [2]:
from src.json_to_pose.base import Pose, PoseAnalyzer
from src.json_to_pose.values import blazepose_lines
import src.utils.plot_painter as plot_painter
from src.utils.process_data import translate_multi_kpt_to_plot_pos
import matplotlib.pyplot as plt
from pathlib import Path

# 參考類型
from matplotlib.axes import Axes
from mpl_toolkits.mplot3d import Axes3D

#### 變數

##### 資料物件

In [3]:
pose = Pose(JSON_FILE)
analyzer = PoseAnalyzer(pose)
saved_folder = Path(SAVE_FOLDER)

##### 使用參數

In [4]:
is_3d = True  # 是否為 3D 姿勢
line_type = blazepose_lines  # 設定連線列表

In [5]:
# 資料範圍
data_range_x = [-1, 1]
data_range_y = [-1, 1]
data_range_z = [0, 2]

# 顏色設定
c_left = "#f00"
c_center = "#0f0"
c_right = "#00f"
c_kpts = "#000"

# 關鍵點大小
dot_size = 5

# 視角
elev, azim, roll = 20, 270, 0

##### 計算資料

In [6]:
# 交錯角度
staggered_angles = analyzer.get_all_pose_shoulder_hip_staggered_angle(is_3d)
staggered_angles_xz = analyzer.get_all_pose_shoulder_hip_staggered_angle_xz()
# 交錯角度相關資訊
angle_min = min(staggered_angles)
angle_max = max(staggered_angles)
idx_min = staggered_angles.index(angle_min)
idx_max = staggered_angles.index(angle_max)

In [7]:
# 取得關鍵點
kpt_min = pose.get_pose_kpt_positions(idx_min, 0, is_3d)
kpt_max = pose.get_pose_kpt_positions(idx_max, 0, is_3d)
# 取得線條座標點
left_min, center_min, right_min = analyzer.get_pose_line_postions(
    idx_min, 0, blazepose_lines, is_3d
)
left_max, center_max, right_max = analyzer.get_pose_line_postions(
    idx_max, 0, blazepose_lines, is_3d
)
# 轉換所有座標點
kpt_min, left_min, center_min, right_min = translate_multi_kpt_to_plot_pos(
    kpt_min, kpt_min, left_min, center_min, right_min
)
kpt_max, left_max, center_max, right_max = translate_multi_kpt_to_plot_pos(
    kpt_max, kpt_max, left_max, center_max, right_max
)

#### 主程式

##### 建立資料夾

In [8]:
if not saved_folder.exists():
    saved_folder.mkdir(parents=True)

##### 儲存肩臀角度分析圖

In [9]:
fig = plt.figure(figsize=(8, 8))

# 繪製圖表
ax: Axes = fig.add_subplot()
ax.set_title(
    f"""close_60
max angle: {angle_max:.2f}
max angle index: {idx_max}
min angle index: {idx_min}"""
)
ax.plot(range(len(staggered_angles)), staggered_angles, label="angle")
ax.legend()

# 儲存結果
out_path = saved_folder / SAVE_ANGLE_PLOT_NAME
plt.savefig(out_path)
plt.close()

##### 儲存最小及最大肩臀角度對應姿勢圖

In [10]:
fig = plt.figure(figsize=(16, 8))

# 初始化圖表設定
if not is_3d:
    ax_min: Axes = fig.add_subplot(121)
    ax_max: Axes = fig.add_subplot(122)
    plot_painter.set_data_range(data_range_x, data_range_y, ax_min)
    plot_painter.set_data_range(data_range_x, data_range_y, ax_max)
else:
    ax_min: Axes3D = fig.add_subplot(121, projection="3d")
    ax_max: Axes3D = fig.add_subplot(122, projection="3d")
    plot_painter.set_data_range(data_range_x, data_range_y, data_range_z, ax_min)
    plot_painter.set_data_range(data_range_x, data_range_y, data_range_z, ax_max)
    ax_min.view_init(elev, azim, roll)
    ax_max.view_init(elev, azim, roll)

# 設定標題
ax_min.set_title("min angle pose")
ax_max.set_title("max angle pose")

# 繪製最小肩臀角度對應姿勢
plot_painter.draw_dots(kpt_min, is_3d, dot_size, c_kpts, "", ax_min)
plot_painter.draw_lines(left_min, is_3d, c_left, ax_min)
plot_painter.draw_lines(center_min, is_3d, c_center, ax_min)
plot_painter.draw_lines(right_min, is_3d, c_right, ax_min)

# 繪製最大肩臀角度對應姿勢
plot_painter.draw_dots(kpt_max, is_3d, dot_size, c_kpts, "", ax_max)
plot_painter.draw_lines(left_max, is_3d, c_left, ax_max)
plot_painter.draw_lines(center_max, is_3d, c_center, ax_max)
plot_painter.draw_lines(right_max, is_3d, c_right, ax_max)

# 儲存結果
out_path = saved_folder / SAVE_POSES_PLOT_NAME
plt.savefig(out_path)
plt.close()